<a href="https://colab.research.google.com/github/shammika001/CB010624/blob/main/_serenity_model_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install muspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 21.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.3 MB/s eta 0:00:00
  Created wheel for pretty-midi: filename=pretty_midi-0.2.10-py3-none-any.whl size=5592288 sha256=69291321c1305528632c5feaad318d0ba2bf33181308f163af352d7b37e643cb
  Stored in directory: /root/.cache/pip/wheels/cd/a5/30/7b8b7f58709f5150f67f98fde4b891ebf0be9ef07a8af49f25
Successfully built pretty-midi
  Attempting uninstall: packaging
    Found existing installation: packaging 24.1
    Uninstalling packaging-24.1:
      Successfully uninstalled packaging-24.1


In [2]:
!pip install pretty_midi

In [8]:
data_dir = "data/emopia/"  # Adjust the path if you prefer a different location

# Load the EMOPIA dataset
emopia = muspy.datasets.EMOPIADataset(root=data_dir, download_and_extract=True)

62988951552it [00:00, 197852939993.93it/s]


Successfully downloaded source : /content/data/emopia/EMOPIA_2.2.zip .
Extracting archive : /content/data/emopia/EMOPIA_2.2.zip ...
Successfully extracted archive : /content/data/emopia .


In [18]:
import muspy
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pretty_midi


In [22]:
# Define a function to print the directory structure
def print_directory_structure(start_path):
    for root, dirs, files in os.walk(start_path):
        level = root.replace(start_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")

# Print the structure of the extracted folder to locate the MIDI files
extract_path = '/data/emopia'
print_directory_structure(extract_path)

# Define functions to load labels and parse MIDI files
def load_labels(label_file_path):
    """Load the labels from the CSV file and return a dictionary mapping IDs to labels."""
    labels_df = pd.read_csv(label_file_path)
    labels_dict = labels_df.set_index('ID')['4Q'].to_dict()
    return labels_dict

def parse_midi(file_path):
    """Parse a MIDI file and convert it to a sequence of tokens."""
    midi_data = pretty_midi.PrettyMIDI(file_path)
    notes = []
    for instrument in midi_data.instruments:
        for note in instrument.notes:
            notes.append(note.pitch)
    return notes

def prepare_dataset(label_file_path, midi_folder_path, max_length):
    """Prepare the dataset by parsing MIDI files and loading labels."""
    labels_dict = load_labels(label_file_path)
    sequences = []
    labels = []

    for midi_id, label in labels_dict.items():
        midi_file_path = os.path.join(midi_folder_path, f"{midi_id}.mid")
        if os.path.exists(midi_file_path):
            tokens = parse_midi(midi_file_path)
            if len(tokens) > max_length:
                tokens = tokens[:max_length]
            sequences.append(tokens)
            labels.append(int(label) - 1)  # Converting 1,2,3,4 to 0,1,2,3 for model input

    return sequences, labels

# Load the label file and specify the directory containing MIDI files
label_file_path = os.path.join(extract_path, '/content/data/emopia/EMOPIA_2.2/label.csv')
midi_folder_path = os.path.join(extract_path, '/content/data/emopia/EMOPIA_2.2/midis')
file_path = os.path.join(extract_path, '/content/data/emopia/EMOPIA_2.2/midis/Q1_0vLPYiPN7qY_0.mid')
  # Adjust this if the MIDI files are in a different directory

# Prepare the dataset
max_seq_length = 512
sequences, labels = prepare_dataset(label_file_path, midi_folder_path, max_seq_length)


In [23]:
# Define the dataset class
class MIDIDataset(Dataset):
    def __init__(self, sequences, labels, max_length):
        self.sequences = sequences
        self.labels = labels
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        label = self.labels[idx]
        # Padding sequence to max_length
        padded_sequence = sequence + [0] * (self.max_length - len(sequence))
        return torch.tensor(padded_sequence, dtype=torch.long), torch.tensor(label, dtype=torch.long)


In [28]:
class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers, dim_feedforward, max_seq_length, num_emotions):
        super(MusicTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Embedding(max_seq_length, d_model)
        self.emotion_embedding = nn.Embedding(num_emotions, d_model)
        self.transformer = nn.Transformer(d_model, nhead, num_encoder_layers, num_encoder_layers, dim_feedforward, batch_first=True)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(0.1)
        self.d_model = d_model

    def forward(self, src, src_mask, emotion):
        src = self.embedding(src) * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))
        src = src + self.pos_encoder(torch.arange(0, src.size(1)).unsqueeze(0).to(src.device))
        emotion = self.emotion_embedding(emotion).unsqueeze(1)
        src = src + emotion
        output = self.transformer(src, src, src_mask)
        output = self.fc_out(self.dropout(output))
        return output

    def generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask

In [29]:
def train_model(model, dataloader, criterion, optimizer, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            batch_size, seq_len = inputs.size()
            src_mask = model.generate_square_subsequent_mask(seq_len).to(inputs.device)
            outputs = model(inputs, src_mask, labels)
            loss = criterion(outputs.view(-1, outputs.size(-1)), inputs.view(-1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f'Epoch {epoch + 1}, Loss: {epoch_loss / len(dataloader)}')
device = '/content/data/device'

In [26]:
def generate_music(model, start_sequence, emotion, max_length, temperature=1.0):
    model.eval()
    generated = start_sequence
    src = torch.tensor(generated, dtype=torch.long).unsqueeze(0).to(device)
    emotion = torch.tensor([emotion], dtype=torch.long).to(device)

    for _ in range(max_length - len(start_sequence)):
        src_mask = model.generate_square_subsequent_mask(src.size(1)).to(src.device)
        with torch.no_grad():
            output = model(src, src_mask, emotion)
        next_token = torch.multinomial(torch.softmax(output[0, -1, :] / temperature, dim=-1), 1)
        generated.append(next_token.item())
        src = torch.tensor(generated, dtype=torch.long).unsqueeze(0).to(device)

    return generated


In [ ]:
vocab_size = 128  # Example vocabulary size (e.g., number of MIDI pitches)
d_model = 512
nhead = 8
num_encoder_layers = 6
dim_feedforward = 2048
max_seq_length = 512
num_emotions = 4
batch_size = 32
num_epochs = 10

# Create dataset and dataloader
dataset = MIDIDataset(sequences, labels, max_seq_length)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model, criterion, and optimizer
model = MusicTransformer(vocab_size, d_model, nhead, num_encoder_layers, dim_feedforward, max_seq_length, num_emotions).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
train_model(model, dataloader, criterion, optimizer, num_epochs)

# Generate music
start_sequence = [60, 62, 64, 65, 67]  # Example starting sequence (e.g., MIDI pitches)
emotion = 1  # Example emotion label (e.g., Q2)
generated_sequence = generate_music(model, start_sequence, emotion, max_seq_length)
print(f"Generated sequence: {generated_sequence}")